***

Preparing Workspace

***

In [ ]:
# Packages
import pandas as pd
import numpy as np
import json
import requests
import os
from functools import reduce
from tqdm import tqdm
import functools as ft
import math
import time
pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'BLS')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
path_code    = os.path.join(path_git, 'Data', 'BLS')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Data', 'BLS', 'config')

print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0,     'Functions.py')).read())
exec(open(os.path.join(path_config , 'BLS Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
exec(open(os.path.join(path_config, 'api_key.txt')).read())
api_key = dict_api[user]

***

Preparing Imports

***

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())

***

Importing

***

Version 2 (registered API key) allows us to pull:  50 Series ID's per request, 20 years of data per request, 500 requests per day

In [ ]:
# Execute script to import BLS Data
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 01b - Run API Queries.py')).read())

***

Exporting

***

In [ ]:
# Export
export_title = '_'.join([indicator_name, geography, 'BLS']) + '_raw.csv'
print("Exporting " + export_title + " to the following location: ")
print(path_raw)
df_bls_raw.to_csv(os.path.join(path_raw, export_title), index = False)

print('')
print('Successfully exported!')

***

Processing (optional)

***

In [ ]:
df_bls = df_bls_raw.copy()

df_bls = pd.melt(df_bls, id_vars = ['year', 'periodName'], var_name = 'seriesID', value_name = 'value')    
df_bls['date_'] = df_bls['year'].astype('str') + '-' + df_bls['periodName'].astype('str')
df_bls['date_'] = pd.to_datetime(df_bls['date_'])
df_bls['value'] = df_bls['value'].astype('float32').apply(lambda x: x*1000)
df_bls = df_bls.merge(df_series_area, on = 'seriesID')
df_bls = df_bls.sort_values(['area_text', 'industry_name', 'date_'], ascending = [True, True, False])

df_bls